In [2]:
from pathlib import Path

CASE_DIR = Path(r"C:\Users\misaz\lab1\case_08")

print("Папка существует:", CASE_DIR.exists())
print("\nФайлы внутри папки:")

for file in CASE_DIR.iterdir():
    print(file.name)

Папка существует: True

Файлы внутри папки:
.ipynb_checkpoints
alibi.json
case.json
lab1_case_08.ipynb.ipynb
README.txt
statements.json
suspects.csv
visits_backup.csv
visits_evening.csv
visits_morning.csv


In [3]:
import pandas as pd
from sqlalchemy import create_engine
from pathlib import Path

# Путь к материалам нашей справы
CASE_DIR = Path(r"C:\Users\misaz\lab1\case_08")

# Создаём SQLite-базу данных
engine = create_engine(
    "sqlite:///C:/Users/misaz/lab1/case_08.db"
)

# Загружаем четыре CSV-файла
suspects = pd.read_csv(CASE_DIR / "suspects.csv")
visits_morning = pd.read_csv(CASE_DIR / "visits_morning.csv")
visits_evening = pd.read_csv(CASE_DIR / "visits_evening.csv")
visits_backup = pd.read_csv(CASE_DIR / "visits_backup.csv")

# Сохраняем таблицы в базу данных
suspects.to_sql("suspects", engine, if_exists="replace", index=False)
visits_morning.to_sql("visits_morning", engine, if_exists="replace", index=False)
visits_evening.to_sql("visits_evening", engine, if_exists="replace", index=False)
visits_backup.to_sql("visits_backup", engine, if_exists="replace", index=False)

print("Данные успешно загружены в SQLite!")

print("\nПодозреваемые:")
display(suspects.head())

print("\nУтренняя смена:")
display(visits_morning.head())

print("\nВечерняя смена:")
display(visits_evening.head())

print("\nРезервный журнал:")
display(visits_backup.head())

Данные успешно загружены в SQLite!

Подозреваемые:


,suspect_id,name,role
0,S01,Ігор Пасічник,реставратор
1,S02,Захар Тесля,екскурсовод
2,S03,Галина Мірошник,електрик
3,S04,Марія Снігур,охоронець
4,S05,Ярина Кушнір,науковий співробітник



Утренняя смена:


,visit_id,suspect_id,room,hour,sensor_id
0,V001,S05,Зброярня,9,SEN-536
1,V005,S10,Зброярня,10,SEN-775
2,V007,S06,Гобеленова,12,SEN-142
3,V008,S02,Вітражна,9,SEN-650
4,V011,S07,Єгипетська,12,SEN-639



Вечерняя смена:


,visit_id,suspect_id,room,hour,sensor_id
0,V002,S02,Бароко,15,SEN-518
1,V003,S04,NaN,18,SEN-457
2,V004,S05,Гобеленова,17,SEN-646
3,V006,S05,Антична,15,SEN-351
4,V009,S01,Єгипетська,20,SEN-626



Резервный журнал:


,visit_id,room
0,V003,Жовта
1,V013,Вітражна
2,V027,Вітражна
3,V041,Дзеркальна
4,V900,Бароко


In [4]:
from pymongo import MongoClient

client = MongoClient(
    "mongodb://localhost:27017/",
    serverSelectionTimeoutMS=5000
)

try:
    print("MongoDB подключена!")
    print("Версия MongoDB:", client.server_info()["version"])
except Exception as e:
    print("Ошибка подключения:", e)

MongoDB подключена!
Версия MongoDB: 8.3.13


In [5]:
import json
from pathlib import Path
from pymongo import MongoClient

# Путь к файлам твоего варианта
CASE_DIR = Path(r"C:\Users\misaz\lab1\case_08")

# Подключаемся к MongoDB
client = MongoClient("mongodb://localhost:27017/")
db = client["museum_case_08"]

# Загружаем свидетельства и алиби
with open(CASE_DIR / "statements.json", encoding="utf-8") as file:
    statements_data = json.load(file)

with open(CASE_DIR / "alibi.json", encoding="utf-8") as file:
    alibi_data = json.load(file)

# Очищаем коллекции перед загрузкой
db["statements"].drop()
db["alibi"].drop()

# Загружаем документы
if statements_data:
    db["statements"].insert_many(statements_data)

if alibi_data:
    db["alibi"].insert_many(alibi_data)

print("Свидетельства загружены:", db["statements"].count_documents({}))
print("Записи об алиби загружены:", db["alibi"].count_documents({}))

Свидетельства загружены: 11
Записи об алиби загружены: 6


In [6]:
import pandas as pd
from sqlalchemy import create_engine

# Подключаемся к SQLite
engine = create_engine("sqlite:///C:/Users/misaz/lab1/case_08.db")

# Читаем таблицы из SQLite
suspects_sql = pd.read_sql_query("SELECT * FROM suspects", engine)
morning_sql = pd.read_sql_query("SELECT * FROM visits_morning", engine)
evening_sql = pd.read_sql_query("SELECT * FROM visits_evening", engine)
backup_sql = pd.read_sql_query("SELECT * FROM visits_backup", engine)

print("Подозреваемые из SQLite:")
display(suspects_sql.head())

print("Утренняя смена из SQLite:")
display(morning_sql.head())

print("Вечерняя смена из SQLite:")
display(evening_sql.head())

print("Резервный журнал из SQLite:")
display(backup_sql.head())

# Читаем документы из MongoDB, исключая служебное поле _id
statements = pd.DataFrame(
    list(db["statements"].find({}, {"_id": 0}))
)

alibi = pd.DataFrame(
    list(db["alibi"].find({}, {"_id": 0}))
)

print("Свидетельства из MongoDB:")
display(statements)

print("Алиби из MongoDB:")
display(alibi)

Подозреваемые из SQLite:


,suspect_id,name,role
0,S01,Ігор Пасічник,реставратор
1,S02,Захар Тесля,екскурсовод
2,S03,Галина Мірошник,електрик
3,S04,Марія Снігур,охоронець
4,S05,Ярина Кушнір,науковий співробітник


Утренняя смена из SQLite:


,visit_id,suspect_id,room,hour,sensor_id
0,V001,S05,Зброярня,9,SEN-536
1,V005,S10,Зброярня,10,SEN-775
2,V007,S06,Гобеленова,12,SEN-142
3,V008,S02,Вітражна,9,SEN-650
4,V011,S07,Єгипетська,12,SEN-639


Вечерняя смена из SQLite:


,visit_id,suspect_id,room,hour,sensor_id
0,V002,S02,Бароко,15,SEN-518
1,V003,S04,NaN,18,SEN-457
2,V004,S05,Гобеленова,17,SEN-646
3,V006,S05,Антична,15,SEN-351
4,V009,S01,Єгипетська,20,SEN-626


... (2 outputs left out)

Свидетельства из MongoDB:


,witness,saw_suspect_id,room,hour
0,вахтер Петренко,S02,Жовта,9
1,вахтер Петренко,S04,Гобеленова,13
2,волонтер Сидір,S04,Вітражна,20
3,водій Юрченко,S06,Жовта,19
4,вахтер Петренко,S08,Єгипетська,12
5,турист Новак,S10,Антична,10
6,водій Юрченко,S10,Антична,10
7,турист Новак,X01,Антична,17
8,студентка Гнатюк,X02,Бароко,17
9,волонтер Сидір,X03,Жовта,14


Алиби из MongoDB:


,suspect_id,confirmed_by
0,S01,студентка Гнатюк
1,S03,касирка Олійник
2,S04,студентка Гнатюк
3,S05,вахтер Петренко
4,S07,вахтер Петренко
5,S09,турист Новак


In [7]:
# Делаем копии журналов, чтобы не изменять исходные таблицы
evening_fixed = evening_sql.copy()
backup = backup_sql.copy()

# Используем visit_id как индекс
evening_fixed = evening_fixed.set_index("visit_id")
backup = backup.set_index("visit_id")

# Восстанавливаем пропущенные залы из резервного журнала
rooms_fixed = evening_fixed["room"].combine_first(backup["room"])

# Оставляем только записи, которые были в вечерней смене
rooms_fixed = rooms_fixed.loc[evening_fixed.index]

# Возвращаем восстановленные залы в таблицу
evening_fixed["room"] = rooms_fixed

# Возвращаем visit_id из индекса в обычную колонку
evening_fixed = evening_fixed.reset_index()

# Восстанавливаем целочисленный тип столбца hour
evening_fixed["hour"] = evening_fixed["hour"].astype(int)

print("Вечерняя смена после восстановления залов:")
display(evening_fixed)

Вечерняя смена после восстановления залов:


,visit_id,suspect_id,room,hour,sensor_id
0,V002,S02,Бароко,15,SEN-518
1,V003,S04,Жовта,18,SEN-457
2,V004,S05,Гобеленова,17,SEN-646
3,V006,S05,Антична,15,SEN-351
4,V009,S01,Єгипетська,20,SEN-626
5,V010,S07,Єгипетська,18,SEN-559
6,V012,S06,Антична,15,SEN-710
7,V013,S05,Вітражна,14,SEN-158
8,V014,S07,Єгипетська,17,SEN-599
9,V015,S10,Єгипетська,15,SEN-484


In [8]:
# Объединяем вечерний и утренний журналы
visits_all = pd.concat(
    [evening_fixed, morning_sql],
    ignore_index=True
)

# Убираем служебный столбец sensor_id
visits_all = visits_all.drop(columns=["sensor_id"])

# Возвращаем visit_id в обычную колонку, если это необходимо
if "visit_id" not in visits_all.columns:
    visits_all = visits_all.reset_index()

print("Количество записей в объединённом журнале:", len(visits_all))
print("\nОбъединённый журнал:")
display(visits_all)

Количество записей в объединённом журнале: 52

Объединённый журнал:


,visit_id,suspect_id,room,hour
0,V002,S02,Бароко,15
1,V003,S04,Жовта,18
2,V004,S05,Гобеленова,17
3,V006,S05,Антична,15
4,V009,S01,Єгипетська,20
5,V010,S07,Єгипетська,18
6,V012,S06,Антична,15
7,V013,S05,Вітражна,14
8,V014,S07,Єгипетська,17
9,V015,S10,Єгипетська,15


In [9]:
# Присоединяем справочник сотрудников к журналу
visits_named = visits_all.merge(
    suspects_sql,
    on="suspect_id",
    how="left"
)

print("Журнал с именами сотрудников:")
display(visits_named)

Журнал с именами сотрудников:


,visit_id,suspect_id,room,hour,name,role
0,V002,S02,Бароко,15,Захар Тесля,екскурсовод
1,V003,S04,Жовта,18,Марія Снігур,охоронець
2,V004,S05,Гобеленова,17,Ярина Кушнір,науковий співробітник
3,V006,S05,Антична,15,Ярина Кушнір,науковий співробітник
4,V009,S01,Єгипетська,20,Ігор Пасічник,реставратор
5,V010,S07,Єгипетська,18,Раїса Дубова,фотограф
6,V012,S06,Антична,15,Устим Романко,прибиральник
7,V013,S05,Вітражна,14,Ярина Кушнір,науковий співробітник
8,V014,S07,Єгипетська,17,Раїса Дубова,фотограф
9,V015,S10,Єгипетська,15,Андрій Лозовий,екскурсовод


In [10]:
# Сравниваем четыре типа объединения свидетельств со справочником

merge_inner = statements.merge(
    suspects_sql,
    left_on="saw_suspect_id",
    right_on="suspect_id",
    how="inner"
)

merge_left = statements.merge(
    suspects_sql,
    left_on="saw_suspect_id",
    right_on="suspect_id",
    how="left"
)

merge_right = statements.merge(
    suspects_sql,
    left_on="saw_suspect_id",
    right_on="suspect_id",
    how="right"
)

merge_outer = statements.merge(
    suspects_sql,
    left_on="saw_suspect_id",
    right_on="suspect_id",
    how="outer"
)

print("Количество строк при inner:", len(merge_inner))
print("Количество строк при left:", len(merge_left))
print("Количество строк при right:", len(merge_right))
print("Количество строк при outer:", len(merge_outer))

print("\nПример результата outer:")
display(merge_outer)

Количество строк при inner: 7
Количество строк при left: 11
Количество строк при right: 12
Количество строк при outer: 16

Пример результата outer:


,witness,saw_suspect_id,room,hour,suspect_id,name,role
0,NaN,NaN,NaN,NaN,S01,Ігор Пасічник,реставратор
1,вахтер Петренко,S02,Жовта,9.0,S02,Захар Тесля,екскурсовод
2,NaN,NaN,NaN,NaN,S03,Галина Мірошник,електрик
3,вахтер Петренко,S04,Гобеленова,13.0,S04,Марія Снігур,охоронець
4,волонтер Сидір,S04,Вітражна,20.0,S04,Марія Снігур,охоронець
5,NaN,NaN,NaN,NaN,S05,Ярина Кушнір,науковий співробітник
6,водій Юрченко,S06,Жовта,19.0,S06,Устим Романко,прибиральник
7,NaN,NaN,NaN,NaN,S07,Раїса Дубова,фотограф
8,вахтер Петренко,S08,Єгипетська,12.0,S08,Еліна Бортник,доглядач
9,NaN,NaN,NaN,NaN,S09,Павло Чумак,реставратор


In [11]:
try:
    pivot_result = visits_named.pivot(
        index="name",
        columns="room",
        values="visit_id"
    )
    display(pivot_result)
except ValueError as e:
    print("Помилка pivot:", e)

Помилка pivot: Index contains duplicate entries, cannot reshape


In [12]:
pivot_table_result = visits_named.pivot_table(
    index="name",
    columns="room",
    values="visit_id",
    aggfunc="count",
    fill_value=0
)

display(pivot_table_result)

room,Єгипетська,Антична,Бароко,Вітражна,Гобеленова,Дзеркальна,Жовта,Зброярня
name,,,,,,,,
Ігор Пасічник,1,0,1,1,1,0,0,0
Андрій Лозовий,1,1,0,2,0,0,0,1
Галина Мірошник,0,2,0,0,0,1,2,0
Еліна Бортник,1,0,0,0,2,0,0,0
Захар Тесля,1,0,1,1,1,1,1,0
Марія Снігур,1,0,0,1,1,0,1,0
Павло Чумак,1,3,0,0,0,0,1,0
Раїса Дубова,3,3,0,0,0,0,0,1
Устим Романко,0,1,0,0,1,0,2,2


In [13]:
with open(CASE_DIR / "case.json", encoding="utf-8") as f:
    case_data = json.load(f)

print(case_data)

{'n': 8, 'crime_room': 'Антична', 'crime_hour': 15}


In [14]:
crime_room = case_data["crime_room"]
crime_hour = case_data["crime_hour"]

suspects_at_crime = visits_named[
    (visits_named["room"] == crime_room) &
    (visits_named["hour"] == crime_hour)
]

display(suspects_at_crime[["visit_id", "suspect_id", "name", "room", "hour"]])

,visit_id,suspect_id,name,room,hour
3,V006,S05,Ярина Кушнір,Антична,15
6,V012,S06,Устим Романко,Антична,15
19,V046,S07,Раїса Дубова,Антична,15


In [15]:
alibi_ids = set(alibi["suspect_id"])

suspects_at_crime = suspects_at_crime.copy()
suspects_at_crime["has_alibi"] = suspects_at_crime["suspect_id"].isin(alibi_ids)

display(suspects_at_crime[["suspect_id", "name", "has_alibi"]])

,suspect_id,name,has_alibi
3,S05,Ярина Кушнір,True
6,S06,Устим Романко,False
19,S07,Раїса Дубова,True


In [17]:
print("Винний:")
display(guilty[["suspect_id", "name", "role"]])

Винний:


,suspect_id,name,role
6,S06,Устим Романко,прибиральник
